# Import

In [1]:
import pandas as pd

import requests
from bs4 import BeautifulSoup

from tqdm import tqdm

# Get Url & Height

In [2]:
lst = []

for table in tqdm(range(0, 8)):
    url = 'https://en.wikipedia.org/wiki/List_of_mountains_by_elevation'

    response = requests.get(url).content
    soup = BeautifulSoup(response, 'lxml')

    table = soup.find_all('table')[table]

    for row in table.find_all('tr')[1:]:
        height = row.find_all('td')[1].text

        cells = row.find_all('td')[0]

        for link in cells.find_all('a', limit=1):
            url = link.get('href')

            url_height = [url, height]

            lst.append(url_height)

100%|██████████| 8/8 [00:04<00:00,  1.76it/s]


In [3]:
print(lst[:10])

[['/wiki/Mount_Everest', '8,848'], ['/wiki/K2', '8,611'], ['/wiki/Kangchenjunga', '8,586'], ['/wiki/Lhotse', '8,516'], ['/wiki/Makalu', '8,485'], ['/wiki/Cho_Oyu', '8,201'], ['/wiki/Dhaulagiri', '8,167'], ['/wiki/Manaslu', '8,163'], ['/wiki/Nanga_Parbat', '8,126'], ['/wiki/Annapurna', '8,091']]


In [4]:
df = pd.DataFrame(lst, columns=['Url', 'Height'])

df

,Url,Height
0,/wiki/Mount_Everest,"8,848"
1,/wiki/K2,"8,611"
2,/wiki/Kangchenjunga,"8,586"
3,/wiki/Lhotse,"8,516"
4,/wiki/Makalu,"8,485"
...,...,...
1422,/wiki/Buachaille_Etive_Mor,"1,022"
1423,/wiki/Munboksan,"1,015"
1424,/wiki/K%C3%A9kes,"1,014"
1425,/wiki/Mount_Belumut,"1,010"


In [5]:
df['Empty'] = df['Url'].str.startswith('/wiki/')

df

,Url,Height,Empty
0,/wiki/Mount_Everest,"8,848",True
1,/wiki/K2,"8,611",True
2,/wiki/Kangchenjunga,"8,586",True
3,/wiki/Lhotse,"8,516",True
4,/wiki/Makalu,"8,485",True
...,...,...,...
1422,/wiki/Buachaille_Etive_Mor,"1,022",True
1423,/wiki/Munboksan,"1,015",True
1424,/wiki/K%C3%A9kes,"1,014",True
1425,/wiki/Mount_Belumut,"1,010",True


In [6]:
df.drop(df[df['Empty'] == False].index, inplace=True)

df

,Url,Height,Empty
0,/wiki/Mount_Everest,"8,848",True
1,/wiki/K2,"8,611",True
2,/wiki/Kangchenjunga,"8,586",True
3,/wiki/Lhotse,"8,516",True
4,/wiki/Makalu,"8,485",True
...,...,...,...
1421,/wiki/Girnar,"1,031",True
1422,/wiki/Buachaille_Etive_Mor,"1,022",True
1423,/wiki/Munboksan,"1,015",True
1424,/wiki/K%C3%A9kes,"1,014",True


In [7]:
df.drop(['Empty'], axis=1, inplace=True)

df

,Url,Height
0,/wiki/Mount_Everest,"8,848"
1,/wiki/K2,"8,611"
2,/wiki/Kangchenjunga,"8,586"
3,/wiki/Lhotse,"8,516"
4,/wiki/Makalu,"8,485"
...,...,...
1421,/wiki/Girnar,"1,031"
1422,/wiki/Buachaille_Etive_Mor,"1,022"
1423,/wiki/Munboksan,"1,015"
1424,/wiki/K%C3%A9kes,"1,014"


In [8]:
df['Url'] = 'https://en.wikipedia.org' + df['Url']

df

,Url,Height
0,https://en.wikipedia.org/wiki/Mount_Everest,"8,848"
1,https://en.wikipedia.org/wiki/K2,"8,611"
2,https://en.wikipedia.org/wiki/Kangchenjunga,"8,586"
3,https://en.wikipedia.org/wiki/Lhotse,"8,516"
4,https://en.wikipedia.org/wiki/Makalu,"8,485"
...,...,...
1421,https://en.wikipedia.org/wiki/Girnar,"1,031"
1422,https://en.wikipedia.org/wiki/Buachaille_Etive...,"1,022"
1423,https://en.wikipedia.org/wiki/Munboksan,"1,015"
1424,https://en.wikipedia.org/wiki/K%C3%A9kes,"1,014"


In [9]:
df.reset_index(drop=True, inplace=True)

df

,Url,Height
0,https://en.wikipedia.org/wiki/Mount_Everest,"8,848"
1,https://en.wikipedia.org/wiki/K2,"8,611"
2,https://en.wikipedia.org/wiki/Kangchenjunga,"8,586"
3,https://en.wikipedia.org/wiki/Lhotse,"8,516"
4,https://en.wikipedia.org/wiki/Makalu,"8,485"
...,...,...
1365,https://en.wikipedia.org/wiki/Girnar,"1,031"
1366,https://en.wikipedia.org/wiki/Buachaille_Etive...,"1,022"
1367,https://en.wikipedia.org/wiki/Munboksan,"1,015"
1368,https://en.wikipedia.org/wiki/K%C3%A9kes,"1,014"


# Get Name, Height & Url

In [10]:
urls = df['Url'].tolist()

print(urls[:10])

['https://en.wikipedia.org/wiki/Mount_Everest', 'https://en.wikipedia.org/wiki/K2', 'https://en.wikipedia.org/wiki/Kangchenjunga', 'https://en.wikipedia.org/wiki/Lhotse', 'https://en.wikipedia.org/wiki/Makalu', 'https://en.wikipedia.org/wiki/Cho_Oyu', 'https://en.wikipedia.org/wiki/Dhaulagiri', 'https://en.wikipedia.org/wiki/Manaslu', 'https://en.wikipedia.org/wiki/Nanga_Parbat', 'https://en.wikipedia.org/wiki/Annapurna']


In [11]:
lst = []

for url in tqdm(urls):

    response = requests.get(url).content
    soup = BeautifulSoup(response, 'lxml')

    table = soup.find('table', class_='infobox vcard')

    try:
        name = soup.find('h1', class_='firstHeading').text
    except AttributeError:
        name = None

    try:
        height = table.find('td', style='padding: 0.2em 0em; vertical-align:text-top;').text
    except AttributeError:
        height = None

    try:
        for link in table.find_all('a', class_='external text', limit=1):
            url = link.get('href')
    except AttributeError:
        urls = None

    name_height_url = [name, height, url]

    lst.append(name_height_url)

100%|██████████| 1370/1370 [03:52<00:00,  5.90it/s]


In [12]:
print(lst[:10])

[['Mount Everest', '8,848.86\xa0m (29,031.7\xa0ft)\xa0\u202f[1] Ranked 1st', '//geohack.toolforge.org/geohack.php?pagename=Mount_Everest&params=27_59_17_N_86_55_31_E_type:mountain_scale:100000'], ['K2', '8,610\xa0m (28,250\xa0ft)\xa0\u202fRanked 2nd', '//geohack.toolforge.org/geohack.php?pagename=K2&params=35_52_57_N_76_30_48_E_type:mountain_scale:100000'], ['Kangchenjunga', '8,586\xa0m (28,169\xa0ft)\u202f[1]Ranked 3rd', '//geohack.toolforge.org/geohack.php?pagename=Kangchenjunga&params=27_42_09_N_88_08_48_E_type:mountain_scale:100000'], ['Lhotse', '8,516\xa0m (27,940\xa0ft)\u202f[nb 1]Ranked 4th', '//geohack.toolforge.org/geohack.php?pagename=Lhotse&params=27_57_42_N_86_56_00_E_type:mountain_scale:100000'], ['Makalu', '8,463\xa0m (27,766\xa0ft)\u202f[1][notes 1]Ranked 5th', '//geohack.toolforge.org/geohack.php?pagename=Makalu&params=27_53_23_N_87_05_20_E_type:mountain_region:NP_scale:100000'], ['Cho Oyu', '8,188\xa0m (26,864\xa0ft)\u202fRanked 6th', '//geohack.toolforge.org/geohack.p

In [13]:
df2 = pd.DataFrame(lst, columns=['Name', 'Height 2', 'Url 2'])

df2

,Name,Height 2,Url 2
0,Mount Everest,"8,848.86 m (29,031.7 ft) [1] Ranked 1st",//geohack.toolforge.org/geohack.php?pagename=M...
1,K2,"8,610 m (28,250 ft) Ranked 2nd",//geohack.toolforge.org/geohack.php?pagename=K...
2,Kangchenjunga,"8,586 m (28,169 ft) [1]Ranked 3rd",//geohack.toolforge.org/geohack.php?pagename=K...
3,Lhotse,"8,516 m (27,940 ft) [nb 1]Ranked 4th",//geohack.toolforge.org/geohack.php?pagename=L...
4,Makalu,"8,463 m (27,766 ft) [1][notes 1]Ranked 5th",//geohack.toolforge.org/geohack.php?pagename=M...
...,...,...,...
1365,Girnar,"1,069 m (3,507 ft)",//geohack.toolforge.org/geohack.php?pagename=G...
1366,Buachaille Etive Mòr,"1,021.4 m (3,351 ft) [1]",//geohack.toolforge.org/geohack.php?pagename=B...
1367,Munboksan,"1,015 m (3,330 ft)",//geohack.toolforge.org/geohack.php?pagename=M...
1368,Kékes,"1,014 m (3,327 ft) [1]",//geohack.toolforge.org/geohack.php?pagename=K...


In [14]:
df

,Url,Height
0,https://en.wikipedia.org/wiki/Mount_Everest,"8,848"
1,https://en.wikipedia.org/wiki/K2,"8,611"
2,https://en.wikipedia.org/wiki/Kangchenjunga,"8,586"
3,https://en.wikipedia.org/wiki/Lhotse,"8,516"
4,https://en.wikipedia.org/wiki/Makalu,"8,485"
...,...,...
1365,https://en.wikipedia.org/wiki/Girnar,"1,031"
1366,https://en.wikipedia.org/wiki/Buachaille_Etive...,"1,022"
1367,https://en.wikipedia.org/wiki/Munboksan,"1,015"
1368,https://en.wikipedia.org/wiki/K%C3%A9kes,"1,014"


In [15]:
df = pd.concat([df, df2], axis=1)

df

,Url,Height,Name,Height 2,Url 2
0,https://en.wikipedia.org/wiki/Mount_Everest,"8,848",Mount Everest,"8,848.86 m (29,031.7 ft) [1] Ranked 1st",//geohack.toolforge.org/geohack.php?pagename=M...
1,https://en.wikipedia.org/wiki/K2,"8,611",K2,"8,610 m (28,250 ft) Ranked 2nd",//geohack.toolforge.org/geohack.php?pagename=K...
2,https://en.wikipedia.org/wiki/Kangchenjunga,"8,586",Kangchenjunga,"8,586 m (28,169 ft) [1]Ranked 3rd",//geohack.toolforge.org/geohack.php?pagename=K...
3,https://en.wikipedia.org/wiki/Lhotse,"8,516",Lhotse,"8,516 m (27,940 ft) [nb 1]Ranked 4th",//geohack.toolforge.org/geohack.php?pagename=L...
4,https://en.wikipedia.org/wiki/Makalu,"8,485",Makalu,"8,463 m (27,766 ft) [1][notes 1]Ranked 5th",//geohack.toolforge.org/geohack.php?pagename=M...
...,...,...,...,...,...
1365,https://en.wikipedia.org/wiki/Girnar,"1,031",Girnar,"1,069 m (3,507 ft)",//geohack.toolforge.org/geohack.php?pagename=G...
1366,https://en.wikipedia.org/wiki/Buachaille_Etive...,"1,022",Buachaille Etive Mòr,"1,021.4 m (3,351 ft) [1]",//geohack.toolforge.org/geohack.php?pagename=B...
1367,https://en.wikipedia.org/wiki/Munboksan,"1,015",Munboksan,"1,015 m (3,330 ft)",//geohack.toolforge.org/geohack.php?pagename=M...
1368,https://en.wikipedia.org/wiki/K%C3%A9kes,"1,014",Kékes,"1,014 m (3,327 ft) [1]",//geohack.toolforge.org/geohack.php?pagename=K...


In [16]:
df.drop(['Url', 'Height 2'], axis=1, inplace=True)

df.rename(columns={'Url 2': 'Url'}, inplace=True)

df = df[['Name', 'Height', 'Url']]

df

,Name,Height,Url
0,Mount Everest,"8,848",//geohack.toolforge.org/geohack.php?pagename=M...
1,K2,"8,611",//geohack.toolforge.org/geohack.php?pagename=K...
2,Kangchenjunga,"8,586",//geohack.toolforge.org/geohack.php?pagename=K...
3,Lhotse,"8,516",//geohack.toolforge.org/geohack.php?pagename=L...
4,Makalu,"8,485",//geohack.toolforge.org/geohack.php?pagename=M...
...,...,...,...
1365,Girnar,"1,031",//geohack.toolforge.org/geohack.php?pagename=G...
1366,Buachaille Etive Mòr,"1,022",//geohack.toolforge.org/geohack.php?pagename=B...
1367,Munboksan,"1,015",//geohack.toolforge.org/geohack.php?pagename=M...
1368,Kékes,"1,014",//geohack.toolforge.org/geohack.php?pagename=K...


In [17]:
df['Height'] = df['Height'].str.replace(',', '')

df

,Name,Height,Url
0,Mount Everest,8848,//geohack.toolforge.org/geohack.php?pagename=M...
1,K2,8611,//geohack.toolforge.org/geohack.php?pagename=K...
2,Kangchenjunga,8586,//geohack.toolforge.org/geohack.php?pagename=K...
3,Lhotse,8516,//geohack.toolforge.org/geohack.php?pagename=L...
4,Makalu,8485,//geohack.toolforge.org/geohack.php?pagename=M...
...,...,...,...
1365,Girnar,1031,//geohack.toolforge.org/geohack.php?pagename=G...
1366,Buachaille Etive Mòr,1022,//geohack.toolforge.org/geohack.php?pagename=B...
1367,Munboksan,1015,//geohack.toolforge.org/geohack.php?pagename=M...
1368,Kékes,1014,//geohack.toolforge.org/geohack.php?pagename=K...


In [18]:
df['Empty'] = df['Url'].str.startswith('//geohack.toolforge.org/')

df

,Name,Height,Url,Empty
0,Mount Everest,8848,//geohack.toolforge.org/geohack.php?pagename=M...,True
1,K2,8611,//geohack.toolforge.org/geohack.php?pagename=K...,True
2,Kangchenjunga,8586,//geohack.toolforge.org/geohack.php?pagename=K...,True
3,Lhotse,8516,//geohack.toolforge.org/geohack.php?pagename=L...,True
4,Makalu,8485,//geohack.toolforge.org/geohack.php?pagename=M...,True
...,...,...,...,...
1365,Girnar,1031,//geohack.toolforge.org/geohack.php?pagename=G...,True
1366,Buachaille Etive Mòr,1022,//geohack.toolforge.org/geohack.php?pagename=B...,True
1367,Munboksan,1015,//geohack.toolforge.org/geohack.php?pagename=M...,True
1368,Kékes,1014,//geohack.toolforge.org/geohack.php?pagename=K...,True


In [19]:
df.drop(df[df['Empty'] == False].index, inplace=True)

df

,Name,Height,Url,Empty
0,Mount Everest,8848,//geohack.toolforge.org/geohack.php?pagename=M...,True
1,K2,8611,//geohack.toolforge.org/geohack.php?pagename=K...,True
2,Kangchenjunga,8586,//geohack.toolforge.org/geohack.php?pagename=K...,True
3,Lhotse,8516,//geohack.toolforge.org/geohack.php?pagename=L...,True
4,Makalu,8485,//geohack.toolforge.org/geohack.php?pagename=M...,True
...,...,...,...,...
1364,Mount Ramon,1037,//geohack.toolforge.org/geohack.php?pagename=M...,True
1365,Girnar,1031,//geohack.toolforge.org/geohack.php?pagename=G...,True
1366,Buachaille Etive Mòr,1022,//geohack.toolforge.org/geohack.php?pagename=B...,True
1367,Munboksan,1015,//geohack.toolforge.org/geohack.php?pagename=M...,True


In [20]:
df.drop(['Empty'], axis=1, inplace=True)

df

,Name,Height,Url
0,Mount Everest,8848,//geohack.toolforge.org/geohack.php?pagename=M...
1,K2,8611,//geohack.toolforge.org/geohack.php?pagename=K...
2,Kangchenjunga,8586,//geohack.toolforge.org/geohack.php?pagename=K...
3,Lhotse,8516,//geohack.toolforge.org/geohack.php?pagename=L...
4,Makalu,8485,//geohack.toolforge.org/geohack.php?pagename=M...
...,...,...,...
1364,Mount Ramon,1037,//geohack.toolforge.org/geohack.php?pagename=M...
1365,Girnar,1031,//geohack.toolforge.org/geohack.php?pagename=G...
1366,Buachaille Etive Mòr,1022,//geohack.toolforge.org/geohack.php?pagename=B...
1367,Munboksan,1015,//geohack.toolforge.org/geohack.php?pagename=M...


In [21]:
df['Url'] = 'https:' + df['Url']

df

,Name,Height,Url
0,Mount Everest,8848,https://geohack.toolforge.org/geohack.php?page...
1,K2,8611,https://geohack.toolforge.org/geohack.php?page...
2,Kangchenjunga,8586,https://geohack.toolforge.org/geohack.php?page...
3,Lhotse,8516,https://geohack.toolforge.org/geohack.php?page...
4,Makalu,8485,https://geohack.toolforge.org/geohack.php?page...
...,...,...,...
1364,Mount Ramon,1037,https://geohack.toolforge.org/geohack.php?page...
1365,Girnar,1031,https://geohack.toolforge.org/geohack.php?page...
1366,Buachaille Etive Mòr,1022,https://geohack.toolforge.org/geohack.php?page...
1367,Munboksan,1015,https://geohack.toolforge.org/geohack.php?page...


In [22]:
df.dropna(inplace=True)

df

,Name,Height,Url
0,Mount Everest,8848,https://geohack.toolforge.org/geohack.php?page...
1,K2,8611,https://geohack.toolforge.org/geohack.php?page...
2,Kangchenjunga,8586,https://geohack.toolforge.org/geohack.php?page...
3,Lhotse,8516,https://geohack.toolforge.org/geohack.php?page...
4,Makalu,8485,https://geohack.toolforge.org/geohack.php?page...
...,...,...,...
1364,Mount Ramon,1037,https://geohack.toolforge.org/geohack.php?page...
1365,Girnar,1031,https://geohack.toolforge.org/geohack.php?page...
1366,Buachaille Etive Mòr,1022,https://geohack.toolforge.org/geohack.php?page...
1367,Munboksan,1015,https://geohack.toolforge.org/geohack.php?page...


In [23]:
df['Height'] = pd.to_numeric(df['Height'])

df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1322 entries, 0 to 1368
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Name    1322 non-null   object 
 1   Height  1322 non-null   float64
 2   Url     1322 non-null   object 
dtypes: float64(1), object(2)
memory usage: 41.3+ KB


In [24]:
df.reset_index(drop=True, inplace=True)

df

,Name,Height,Url
0,Mount Everest,8848.0,https://geohack.toolforge.org/geohack.php?page...
1,K2,8611.0,https://geohack.toolforge.org/geohack.php?page...
2,Kangchenjunga,8586.0,https://geohack.toolforge.org/geohack.php?page...
3,Lhotse,8516.0,https://geohack.toolforge.org/geohack.php?page...
4,Makalu,8485.0,https://geohack.toolforge.org/geohack.php?page...
...,...,...,...
1317,Mount Ramon,1037.0,https://geohack.toolforge.org/geohack.php?page...
1318,Girnar,1031.0,https://geohack.toolforge.org/geohack.php?page...
1319,Buachaille Etive Mòr,1022.0,https://geohack.toolforge.org/geohack.php?page...
1320,Munboksan,1015.0,https://geohack.toolforge.org/geohack.php?page...


# Get Latitude & Longitude

In [25]:
urls = df['Url'].tolist()

print(urls[:10])

['https://geohack.toolforge.org/geohack.php?pagename=Mount_Everest&params=27_59_17_N_86_55_31_E_type:mountain_scale:100000', 'https://geohack.toolforge.org/geohack.php?pagename=K2&params=35_52_57_N_76_30_48_E_type:mountain_scale:100000', 'https://geohack.toolforge.org/geohack.php?pagename=Kangchenjunga&params=27_42_09_N_88_08_48_E_type:mountain_scale:100000', 'https://geohack.toolforge.org/geohack.php?pagename=Lhotse&params=27_57_42_N_86_56_00_E_type:mountain_scale:100000', 'https://geohack.toolforge.org/geohack.php?pagename=Makalu&params=27_53_23_N_87_05_20_E_type:mountain_region:NP_scale:100000', 'https://geohack.toolforge.org/geohack.php?pagename=Cho_Oyu&params=28_05_39_N_86_39_39_E_type:mountain_scale:100000', 'https://geohack.toolforge.org/geohack.php?pagename=Dhaulagiri&params=28_41_54_N_83_29_15_E_type:mountain_region:NP_scale:100000', 'https://geohack.toolforge.org/geohack.php?pagename=Manaslu&params=28_32_58_N_84_33_43_E_type:mountain_region:NP_scale:100000', 'https://geohack.

In [26]:
lst = []

for url in tqdm(urls):

    response = requests.get(url).content
    soup = BeautifulSoup(response, 'lxml')

    try:
        latitude = soup.find('span', class_='latitude').text
    except AttributeError:
        latitude = None

    try:
        longitude = soup.find('span', class_='longitude').text
    except AttributeError:
        longitude = None

    latitude_longitude = [latitude, longitude]

    lst.append(latitude_longitude)

100%|██████████| 1322/1322 [16:45<00:00,  1.31it/s]


In [27]:
print(lst[:10])

[['27.988056', '86.925278'], ['35.8825', '76.513333'], ['27.7025', '88.146667'], ['27.961667', '86.933333'], ['27.889722', '87.088889'], ['28.094167', '86.660833'], ['28.698333', '83.4875'], ['28.549444', '84.561944'], ['35.2375', '74.589167'], ['28.596111', '83.820278']]


In [28]:
df2 = pd.DataFrame(lst, columns=['Latitude', 'Longitude'])

df2

,Latitude,Longitude
0,27.988056,86.925278
1,35.8825,76.513333
2,27.7025,88.146667
3,27.961667,86.933333
4,27.889722,87.088889
...,...,...
1317,30.502917,34.63925
1318,21.494722,70.505556
1319,56.647303,-4.897797
1320,35.676,129.034


In [29]:
df

,Name,Height,Url
0,Mount Everest,8848.0,https://geohack.toolforge.org/geohack.php?page...
1,K2,8611.0,https://geohack.toolforge.org/geohack.php?page...
2,Kangchenjunga,8586.0,https://geohack.toolforge.org/geohack.php?page...
3,Lhotse,8516.0,https://geohack.toolforge.org/geohack.php?page...
4,Makalu,8485.0,https://geohack.toolforge.org/geohack.php?page...
...,...,...,...
1317,Mount Ramon,1037.0,https://geohack.toolforge.org/geohack.php?page...
1318,Girnar,1031.0,https://geohack.toolforge.org/geohack.php?page...
1319,Buachaille Etive Mòr,1022.0,https://geohack.toolforge.org/geohack.php?page...
1320,Munboksan,1015.0,https://geohack.toolforge.org/geohack.php?page...


In [30]:
df = pd.concat([df, df2], axis=1)

df

,Name,Height,Url,Latitude,Longitude
0,Mount Everest,8848.0,https://geohack.toolforge.org/geohack.php?page...,27.988056,86.925278
1,K2,8611.0,https://geohack.toolforge.org/geohack.php?page...,35.8825,76.513333
2,Kangchenjunga,8586.0,https://geohack.toolforge.org/geohack.php?page...,27.7025,88.146667
3,Lhotse,8516.0,https://geohack.toolforge.org/geohack.php?page...,27.961667,86.933333
4,Makalu,8485.0,https://geohack.toolforge.org/geohack.php?page...,27.889722,87.088889
...,...,...,...,...,...
1317,Mount Ramon,1037.0,https://geohack.toolforge.org/geohack.php?page...,30.502917,34.63925
1318,Girnar,1031.0,https://geohack.toolforge.org/geohack.php?page...,21.494722,70.505556
1319,Buachaille Etive Mòr,1022.0,https://geohack.toolforge.org/geohack.php?page...,56.647303,-4.897797
1320,Munboksan,1015.0,https://geohack.toolforge.org/geohack.php?page...,35.676,129.034


In [31]:
df.drop(['Url'], axis=1, inplace=True)

df

,Name,Height,Latitude,Longitude
0,Mount Everest,8848.0,27.988056,86.925278
1,K2,8611.0,35.8825,76.513333
2,Kangchenjunga,8586.0,27.7025,88.146667
3,Lhotse,8516.0,27.961667,86.933333
4,Makalu,8485.0,27.889722,87.088889
...,...,...,...,...
1317,Mount Ramon,1037.0,30.502917,34.63925
1318,Girnar,1031.0,21.494722,70.505556
1319,Buachaille Etive Mòr,1022.0,56.647303,-4.897797
1320,Munboksan,1015.0,35.676,129.034


In [32]:
df.dropna(inplace=True)

df

,Name,Height,Latitude,Longitude
0,Mount Everest,8848.0,27.988056,86.925278
1,K2,8611.0,35.8825,76.513333
2,Kangchenjunga,8586.0,27.7025,88.146667
3,Lhotse,8516.0,27.961667,86.933333
4,Makalu,8485.0,27.889722,87.088889
...,...,...,...,...
1317,Mount Ramon,1037.0,30.502917,34.63925
1318,Girnar,1031.0,21.494722,70.505556
1319,Buachaille Etive Mòr,1022.0,56.647303,-4.897797
1320,Munboksan,1015.0,35.676,129.034


In [33]:
df.reset_index(drop=True, inplace=True)

df

,Name,Height,Latitude,Longitude
0,Mount Everest,8848.0,27.988056,86.925278
1,K2,8611.0,35.8825,76.513333
2,Kangchenjunga,8586.0,27.7025,88.146667
3,Lhotse,8516.0,27.961667,86.933333
4,Makalu,8485.0,27.889722,87.088889
...,...,...,...,...
1317,Mount Ramon,1037.0,30.502917,34.63925
1318,Girnar,1031.0,21.494722,70.505556
1319,Buachaille Etive Mòr,1022.0,56.647303,-4.897797
1320,Munboksan,1015.0,35.676,129.034


# Write File

In [34]:
df.to_csv('../../data/processed/mountains.csv', index=False)